# AnanthiX AI - Jalon 2 : Preprocessing + Training Baseline

**Objectif** : Préparer les données et entraîner ResNet-50 baseline

In [2]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from pathlib import Path
from collections import Counter
import pickle
import tensorflow_datasets as tfds
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')
DATA_PATH = BASE_PATH / 'data'
RESULTS_PATH = BASE_PATH / 'results'
MODELS_PATH = BASE_PATH / 'models'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('Chargement metadata')

metadata_path = DATA_PATH / 'plantvillage_metadata.json'
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

class_names = metadata['class_names']
num_classes = len(class_names)

print(f'Classes: {num_classes}')

print('Chargement du dataset PlantVillage en mémoire')

ds, info = tfds.load('plant_village', with_info=True, as_supervised=True)
train_ds = ds['train']

all_images = []
all_labels = []

print('Lecture du dataset')
for i, (image, label) in enumerate(train_ds):
    all_images.append(image.numpy().astype(np.uint8))
    all_labels.append(label.numpy())

    if (i + 1) % 10000 == 0:
        print(f'{i + 1}/{info.splits["train"].num_examples}')

all_images = np.array(all_images)
all_labels = np.array(all_labels)

print(f'Dataset en RAM: {all_images.shape}')

print('Création splits train/val/test')

train_indices = []
val_indices = []
test_indices = []

for class_idx in range(num_classes):
    class_mask = all_labels == class_idx
    class_indices = np.where(class_mask)[0]

    n = len(class_indices)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)

    train_indices.extend(class_indices[:n_train])
    val_indices.extend(class_indices[n_train:n_train + n_val])
    test_indices.extend(class_indices[n_train + n_val:])

print(f'Train: {len(train_indices)} | Val: {len(val_indices)} | Test: {len(test_indices)}')

print('Création DataLoaders')

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(256, pad_if_needed=True),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

class FastDataset(Dataset):
    def __init__(self, images, labels, indices, transform=None):
        self.images = images
        self.labels = labels
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        image = self.images[actual_idx]
        label = self.labels[actual_idx]

        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        return image, label

batch_size = 48

train_dataset = FastDataset(all_images, all_labels, train_indices, transform=train_transform)
val_dataset = FastDataset(all_images, all_labels, val_indices, transform=val_test_transform)
test_dataset = FastDataset(all_images, all_labels, test_indices, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f'DataLoaders: train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)}')

print('Chargement ResNet-50')

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(2048, num_classes)

for param in model.layer1.parameters():
    param.requires_grad = False
for param in model.layer2.parameters():
    param.requires_grad = False

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Modèle chargé | Trainable params: {trainable:,}')

print('Calcul class weights')

class_counts = np.bincount(all_labels, minlength=num_classes)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * num_classes
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

print(f'Poids min: {class_weights.min():.4f}, max: {class_weights.max():.4f}')
print(f'Ratio: {class_weights.max() / class_weights.min():.1f}x')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print('Configuration: CrossEntropyLoss (weighted) + Adam + Scheduler')

print('Démarrage entraînement\n')

num_epochs = 10
best_val_loss = float('inf')
best_epoch = 0

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = train_loss / len(train_loader)
    train_acc = 100 * train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save(model.state_dict(), MODELS_PATH / 'resnet50_baseline_weighted.pth')
        print(f'Epoch {epoch+1:2d}/10 | TL: {train_loss:.4f} ({train_acc:5.1f}%) | VL: {val_loss:.4f} ({val_acc:5.1f}%) | SAVED')
    else:
        print(f'Epoch {epoch+1:2d}/10 | TL: {train_loss:.4f} ({train_acc:5.1f}%) | VL: {val_loss:.4f} ({val_acc:5.1f}%)')

print(f'Training completed | Best model at epoch {best_epoch}')

print('Évaluation test set')

model.eval()
test_correct = 0
test_total = 0
all_preds = []
all_labels_test = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels_test.extend(labels.cpu().numpy())

test_acc = 100 * test_correct / test_total
f1_macro = f1_score(all_labels_test, all_preds, average='macro', zero_division=0)
f1_weighted = f1_score(all_labels_test, all_preds, average='weighted', zero_division=0)
precision = precision_score(all_labels_test, all_preds, average='macro', zero_division=0)
recall = recall_score(all_labels_test, all_preds, average='macro', zero_division=0)

cm = confusion_matrix(all_labels_test, all_preds)

print()
print('TEST RESULTS:')
print(f'  Accuracy: {test_acc:.2f}%')
print(f'  F1 Macro: {f1_macro:.4f}')
print(f'  F1 Weighted: {f1_weighted:.4f}')
print(f'  Precision: {precision:.4f}')
print(f'  Recall: {recall:.4f}')

print('Sauvegarde dans Google Drive')

with open(RESULTS_PATH / 'training_history_weighted.json', 'w') as f:
    json.dump(history, f)

metrics = {
    'test_accuracy': float(test_acc),
    'f1_macro': float(f1_macro),
    'f1_weighted': float(f1_weighted),
    'precision': float(precision),
    'recall': float(recall),
    'epochs_trained': len(history['train_loss']),
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'class_weights_applied': True
}

with open(RESULTS_PATH / 'metrics_baseline_weighted.json', 'w') as f:
    json.dump(metrics, f, indent=2)

with open(RESULTS_PATH / 'confusion_matrix_weighted.pkl', 'wb') as f:
    pickle.dump(cm, f)

print(f'Model: {MODELS_PATH / "resnet50_baseline_weighted.pth"}')
print(f'Metrics: {RESULTS_PATH / "metrics_baseline_weighted.json"}')
print(f'History: {RESULTS_PATH / "training_history_weighted.json"}')

print('Génération visualisations')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', marker='s', linewidth=2)
axes[0].axvline(x=best_epoch-1, color='red', linestyle='--', label=f'Best (epoch {best_epoch})')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', marker='o', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', marker='s', linewidth=2)
axes[1].axvline(x=best_epoch-1, color='red', linestyle='--', label=f'Best (epoch {best_epoch})')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = RESULTS_PATH / 'training_curves_baseline_weighted.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.close()

print(f'Curves: {curves_path}')
print('ÉTAPE 2 COMPLETED')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Chargement metadata
Classes: 38
Chargement du dataset PlantVillage en mémoire
Lecture du dataset
10000/54303
20000/54303
30000/54303
40000/54303
50000/54303
Dataset en RAM: (54303, 256, 256, 3)
Création splits train/val/test
Train: 37995 | Val: 8129 | Test: 8179
Création DataLoaders
DataLoaders: train 792 | val 170 | test 171
Chargement ResNet-50
Modèle chargé | Trainable params: 22,150,502
Calcul class weights
Poids min: 0.1427, max: 5.1699
Ratio: 36.2x
Configuration: CrossEntropyLoss (weighted) + Adam + Scheduler
Démarrage entraînement

Epoch  1/10 | TL: 0.4337 ( 91.1%) | VL: 0.0391 ( 98.9%) | SAVED
Epoch  2/10 | TL: 0.0620 ( 98.4%) | VL: 0.0256 ( 99.2%) | SAVED
Epoch  3/10 | TL: 0.0374 ( 98.9%) | VL: 0.0303 ( 99.0%)
Epoch  4/10 | TL: 0.0279 ( 99.2%) | VL: 0.0163 ( 99.5%) | SAVED
Epoch  5/10 | TL: 0.0277 ( 99.2%) | VL: 0.0148 ( 99.6%) | SAVED
E